# Week 2 Day 2 Lab - Agentic RAG, MCP and Memory DeepEval

## Giving the agent a long-term memory

Another 25 claims have arrived at UnderCover, and most of them turn on something you won't find in the claim: when a storm was named, what a device is, or how deep a dive site goes.

We give an agent a long-term memory: Qdrant, a vector database, running as an MCP server. We teach it the facts it's missing, then run the UnderCover agent with and without that memory as experiments in DeepEval.

## 4 parts:

PART 1: The even harder claims

PART 2: A memory over MCP

PART 3: Teaching the agent

PART 4: Experiments in DeepEval

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you'd rather not spend anything, there's a free version of this lab in <a href="week2day2free.ipynb">week2day2free.ipynb</a>.<br/><br/>
            Teaching the agent takes about 3 minutes, and each experiment takes 3 or 4 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
import plotly.express as px
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.evaluate.configs import AsyncConfig
from deepeval.integrations.langchain import CallbackHandler
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
from deepeval.tracing import observe, update_current_trace
from deepeval.tracing.context import current_span_context, current_trace_context
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file in the `undercover` folder.


In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")


OpenRouter API key found


## Part 1: The even harder claims

The new claims are in `data/evals/even_harder`, with their policy records and the handlers' decisions.

In [3]:
EVEN_HARDER = Path("../data/evals/even_harder")


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

25 claims, 25 labels, 25 policies


In [4]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,amount_claimed,decision,payout
0,CLM-1301,Bahamas trip canceled when Hurricane Marisol s...,1835.00,decline,0.00
1,CLM-1302,Cancun vacation canceled when Hurricane Tobias...,3197.00,approve,2473.00
2,CLM-1303,Stuck in Lisbon for a day by an air traffic co...,282.30,decline,0.00
3,CLM-1304,Flight to Denver canceled by the Meridian cabi...,734.40,approve,300.00
4,CLM-1305,"Ferry to Dublin canceled by Storm Ysolde, two ...",432.30,decline,0.00
5,CLM-1306,Train to Rome canceled by the national rail st...,605.60,approve,427.60
6,CLM-1307,Checked bag lost on the way to Tokyo,1611.95,approve,562.95
7,CLM-1308,Robbed outside a metro station in Barcelona,1329.00,approve,1239.00
8,CLM-1309,Backpack stolen on the Paris metro,771.99,approve,671.99
9,CLM-1310,Rented apartment in Split broken into,833.00,approve,434.00


### A lost bag in Tokyo

Here's a claim on a policy with Electronics Plus, which raises the limit for phones, tablets, laptops, cameras, e-readers and smartwatches. What is a Halden Arc?

In [4]:
tokyo = [claim for claim in claims if claim["claim_id"] == "CLM-1307"][0]

print(tokyo["story"])
pd.DataFrame(tokyo["items"])

My checked bag never arrived in Tokyo. Skylark searched for a month, declared it lost and paid me $400. Everything in it is listed, with receipts for anything over $100.


,description,category,amount
0,Suitcase,luggage,189.00
1,Halden Arc,electronics,899.00
2,Lightweight rain jacket,clothing,165.00
3,Two merino T-shirts,clothing,118.00
4,"Hiking shoes, one pair",clothing,139.95
5,Packing cubes,luggage,54.20
6,Charging cables and adapters,electronics,46.80


This is how the handler decided it.

In [5]:
decision = labels["CLM-1307"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

The carrier declared the bag lost. The Halden Arc is a pocket projector, which is not on the Electronics Plus list, so it counts at the $300 per-item limit on Premium, and the other lines are within the limit. The items come to $1,012.95, less the carrier's $400 and the $50 deductible, leaving $562.95, inside the $1,500 maximum.


,step,amount
0,Suitcase,189.00
1,"Halden Arc pocket projector, limited to $300.00",300.00
2,Lightweight rain jacket,165.00
3,Two merino T-shirts,118.00
4,Hiking shoes,139.95
5,Packing cubes,54.20
6,Charging cables and adapters,46.80
7,Less what others have paid or refunded,-400.00
8,Less the deductible,-50.00


### The knowledge base

The facts the handlers used are in `data/knowledge.json`: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators. Some of them aren't needed by any claim.

In [6]:
knowledge = json.loads(Path("../data/knowledge.json").read_text(encoding="utf-8"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(knowledge)

,topic,fact
0,events,"National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east of the Leeward Islands. It is forecast to become a hurricane on July 11 and to pass over the Bahamas on July 12 and 13."
1,events,"National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in the western Caribbean. It is expected to become a hurricane and to reach the coast of the Yucatan Peninsula near Cancun on August 21."
2,events,"Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 hour strike starting at midnight on July 14. Most flights to and from Lisbon and Porto are expected to be canceled on July 14 and 15."
3,events,"Meridian Air press release, August 5, 2026: The union representing our cabin crew has called a strike for August 12 and 13. We expect to cancel about half of our flights on those two days and will rebook affected passengers."
4,events,"North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named. Gales are expected across Ireland, Wales and western Scotland from the morning of September 4, and ferry crossings on the Irish Sea are likely to be canceled."
5,events,"Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail strike from 9 pm on July 21 to 9 pm on July 22. Most high-speed and regional trains will not run."
6,events,"National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the southern Gulf of Mexico and is expected to come ashore in Tamaulipas, Mexico, on July 2."
7,events,"Northwind Airlines press release, July 15, 2026: The pilots' strike planned for July 24 has been called off after an agreement with the union. All flights will operate as scheduled."
8,events,"Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass close to Okinawa on August 27 and 28. Flights and ferries in the Ryukyu Islands are likely to be suspended."
9,events,"Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for August 27. Ferries between Piraeus and the islands will stay in port."


## Part 2: A memory over MCP

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">MCP</h2>
            <span style="color:#118AB2;">MCP, the Model Context Protocol, is an open standard for connecting agents to tools. An MCP server offers tools, and an MCP client makes them available to an agent.<br/><br/>
            Qdrant's MCP server has two tools: <code>qdrant-store</code> saves a piece of text in a vector database, and <code>qdrant-find</code> searches it by meaning. We run it on our own machine with <code>uvx</code>, and the database lives in a local folder.
            </span>
        </td>
    </tr>
</table>

### One adjustment for Windows

When Python launches another program, it hands that program a place to write its error messages. Inside a Jupyter notebook on Windows, that place is not a real file, and the library that launches MCP servers trips over it. The cell below points the MCP server's error log somewhere safe before we connect. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [7]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Not Windows, so nothing to do here


### Connecting to the memory server

Qdrant keeps its database in the folder `qdrant_data`. We delete it first, so the agent starts with an empty memory. The first time you connect, `uvx` downloads the server and a small embedding model, which takes about half a minute.

In [8]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [9]:
MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory"}

parameters = {"transport": "stdio", "command": "uvx", "args": ["mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}

client = MultiServerMCPClient({ "memory": parameters})

memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

qdrant-find: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - Access memories for further analysis 
 - Get some personal information about the user

qdrant-store: Keep the memory for later use, when you are asked to remember something.



Each tool call starts the server, runs the tool and stops the server again. Qdrant's local mode lets only one process open the database at a time, so when the agent searches twice at once, one of the searches fails. `ToolRetryMiddleware` tries a failed tool call again after a short wait.

In [33]:
await client.get_tools()


[StructuredTool(name='qdrant-find', description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', args_schema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x11b475c60>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11bc3b880>),
 StructuredTool(name='qdrant-store', description='Keep the memory for later use, when you are asked to remember something.', args_schema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memo

### An agent with a memory

The memory tools come from the server; the instructions tell the agent how to use them.

The MCP tools only run asynchronously so we need to use async python. We give the model an HTTP client that doesn't keep connections open between calls, so a connection is never reused from one run to the next.

In [10]:
MEMORY_INSTRUCTIONS = """You have a long-term memory.
When you are told a fact, store it in your memory word for word.
When you are asked a question, search your memory and answer from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(limits=httpx.Limits(max_keepalive_connections=0))
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5, http_async_client=http_client)

assistant = create_agent(model=luna, tools=memory_tools, system_prompt=MEMORY_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])

`ask` starts a new conversation every time, so the agent can't answer from earlier messages. Whatever it knows next time has to come from its memory.

In [11]:
async def ask(message):
    """Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["messages"][-1].text


print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

I don’t know what kind of device the Halden Arc is.
I don’t know from the information in my memory.


## Part 3: Teaching the agent

Now we tell the agent each fact in the knowledge base, one message at a time. This takes about 3 minutes.

In [12]:
for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]}...")

Told: National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east ...
Told: National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in ...
Told: Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 ho...
Told: Meridian Air press release, August 5, 2026: The union representing our cabin cre...
Told: North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named....
Told: Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail stri...
Told: National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the...
Told: Northwind Airlines press release, July 15, 2026: The pilots' strike planned for ...
Told: Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass ...
Told: Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for Au...
Told: Paris, May 29, 2026: French air traffic controllers have called a strike for Jun...
Told: Corv

We ask the same questions again, each in a new conversation.

In [13]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

The Halden Arc is a pocket projector that can project an image up to 100 inches from a phone or laptop.
Tropical Storm Marisol was named on July 9, 2026.


A tool is still a function, so we can search the memory ourselves. The search works on meaning, so a question can find a fact that shares none of its words. It returns the 10 closest facts, nearest first.

In [14]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-find"][0]

results = await search.ainvoke({"query": "a gadget for watching films on a wall"})
for line in json.loads(results[0]["text"]):
    print(line)

Results for the query 'a gadget for watching films on a wall'
<entry><content>Halden Arc: a pocket projector that throws a picture of up to 100 inches from a phone or laptop. List price $899.</content><metadata></metadata></entry>
<entry><content>Vantor Halo: a titanium smart ring that tracks sleep, heart rate and activity. List price $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 3: an e-reader with a 7 inch color E Ink screen and a battery that lasts six weeks. List price $329.</content><metadata></metadata></entry>
<entry><content>Corvane Lumen X1: the earlier Corvane smartwatch, released in 2024 at a list price of $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 2: the earlier Lumora e-reader, with a 6 inch black and white screen. List price $199.</content><metadata></metadata></entry>
<entry><content>Halden Beam: a portable Bluetooth speaker with a built-in lamp. List price $199.</content><metadata></metadata></entry>
<entry><con

## Part 4: Experiments in DeepEval

### The UnderCover agent

The instructions, the prompt, `record_decision` and the payout tool are the same as on Day 1.


In [15]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [16]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

Both agents run Luna at `low` with the payout tool. The second one also gets `qdrant-find`, so it can search the memory whenever it decides to. This is agentic RAG: the agent chooses when to retrieve and what to search for. Its server runs in read-only mode, so it can search the memory but can't change it.

In [17]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])

['qdrant-find']


In [32]:
memory_search

[StructuredTool(name='qdrant-find', description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', args_schema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x11b475c60>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x113dc9440>)]

In [20]:
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

memory_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

The scores are the ones from Day 1, with one more for retrieval. For the claims that need a fact from the knowledge base, `found_fact` is whether the agent's searches brought that fact back.

In [21]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [22]:
class RightMetric(BaseMetric):
    """Test case metric: 1 when the agent's decision matches the handler's and the payout is within a dollar."""

    def __init__(self):
        self.threshold = 1
        self.async_mode = False

    def measure(self, test_case, *args, **kwargs):
        answer = json.loads(test_case.actual_output)
        expected = json.loads(test_case.expected_output)
        self.score = int(is_right(answer, expected))
        self.success = self.score >= self.threshold
        self.reason = f"Agent: {answer['decision']} ${answer['payout']:,.2f}. Handler: {expected['decision']} ${expected['payout']:,.2f}. Cost of mistake: ${mistake_cost(answer, expected):,.2f}"
        found = found_fact(answer, expected)
        if found is not None:
            self.reason += f". Found the fact: {found}"
        return self.score

    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case)

    def is_successful(self):
        return self.success

    @property
    def __name__(self):
        return "Right"


def score_table(test_cases):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for test_case in test_cases:
        answer = json.loads(test_case.actual_output)
        expected = json.loads(test_case.expected_output)
        rows.append({
            "claim_id": test_case.name,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Goldens, test cases and metrics</h2>
            <span style="color:#118AB2;">A DeepEval <b>golden</b> is a test input with its expected output, and a dataset is a list of goldens. To score an agent, you run it on each golden and put its answer in an <b>LLMTestCase</b>. A <b>metric</b> scores a test case, and <code>evaluate</code> runs the metrics on all the test cases and reports the results.<br/><br/>
            DeepEval has ready-made metrics, many of them judged by an LLM. Ours are exact comparisons with the handlers' decisions, so we write a custom metric by subclassing <code>BaseMetric</code>. The scores for a whole run, such as kappa, we work out ourselves in pandas.
            </span>
        </td>
    </tr>
</table>

### The dataset

We make the even harder claims into a dataset of goldens, with the handler's decision and the facts each claim needs as the expected output. Each golden is named for its claim id.


In [23]:
goldens = []

for claim in claims:
    label = labels[claim["claim_id"]]
    goldens.append(Golden(
        name=claim["claim_id"],
        input=json.dumps(claim),
        expected_output=json.dumps({"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]}),
        additional_metadata={"claim_id": claim["claim_id"], "tiny": label["tiny"]},
    ))

len(goldens)


25

As on Day 1, `TINY = True` keeps 5 of the claims, and `N_TIMES` is the number of times each experiment runs.


In [24]:
TINY = True
N_TIMES = 1

dataset = EvaluationDataset(goldens=goldens)
if TINY:
    dataset = EvaluationDataset(goldens=[golden for golden in goldens if golden.additional_metadata["tiny"]])

print(f"Running {len(dataset.goldens)} of the {len(goldens)} claims")


Running 5 of the 25 claims


### The task and the metric

`decide` uses `ainvoke`, since the memory tools only run asynchronously. Along with the decision, `decide` keeps what the agent's memory searches returned.

`RightMetric` is a DeepEval metric. It scores one test case as 1 when the agent got the claim right, and 0 when it didn't.


In [25]:
async def decide(agent, claim):
    """Run an agent on one claim and return its decision, the cost of its model calls and what its memory searches returned."""
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [CallbackHandler()]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    retrieved = "\n".join(message.text for message in result["messages"] if message.type == "tool" and message.name == "qdrant-find")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost, "retrieved": retrieved}


@observe(type="agent")
async def run_claim(agent, golden):
    """Run an agent on one golden. The trace of this call is scored with the golden's expected output."""
    answer = await decide(agent, json.loads(golden.input))
    test_case = LLMTestCase(
        name=golden.name,
        input=golden.input,
        actual_output=json.dumps(answer),
        expected_output=golden.expected_output,
    )
    update_current_trace(name=golden.name, test_case=test_case)
    return test_case


async def run_experiment(name, agent):
    """Run an agent on the dataset N_TIMES, score each run with DeepEval, and return the claims of every run in one table."""
    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        # A trace left open by an earlier cell or a run that was stopped would break this run
        current_span_context.set(None)
        current_trace_context.set(None)
        test_cases = [await run_claim(agent, golden) for golden in dataset.goldens]
        evaluate(
            test_cases=test_cases,
            metrics=[RightMetric()],
            identifier=f"{name}, run {run}",
            async_config=AsyncConfig(run_async=False),
        )
        tables.append(score_table(test_cases).assign(run=run))
    return pd.concat(tables, ignore_index=True)


### Experiment 1: the payout tool

This is the agent from Day 1 with the payout tool. On Day 1 we turned Luna's reasoning off, and today both agents run with Luna at `low`. `run_experiment` runs an agent on every claim, one after another, as many times as `N_TIMES`, and scores each run with `RightMetric`. Each run takes about 3 minutes.

DeepEval saves the traces of the latest run. To read them, run `uv run deepeval inspect` in the terminal, in the folder you ran the notebook from (`labs`). Each run replaces the one before, so inspect after each experiment.


In [26]:
payout_table = await run_experiment("Payout tool", payout_agent)


CLM-1301: refer $0.00


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

CLM-1307: approve $1,161.95


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

CLM-1314: refer $0.00


...

✨ You're running DeepEval's latest Right Metric! (using None, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ CLM-1301                                                                                                    │
│  ├──   Input:              {"claim_id": "CLM-1301", "policy_id": "POL-2301", "customer_id": "CUS-3301",         │
│  │                         "claim_type": "trip_cancellation", "submitted_on": "2026-07-24", "incident_date":    │
│  │                         "2026-07-11", "incident_location": "Miami, Florida", "summary": "Bahamas trip        │
│  │                         canceled when Hurricane Marisol stopped the flights", "story": "Skylark canceled     │
│  │                         every flight to Nassau from July 12 to July 14 because of Hurricane Marisol, so      │
│  │                         we had to cancel our week at the resort. The resort kept our deposit and the         │
│  │                         airline gave us a voucher we will never use. I am claiming what we lost.",           │
│  │                         "items": [{"description": "Resort deposit, not refundable", "category":              │
│  │                         "trip_cost", "amount": 1450.0}, {"description": "Catamaran and snorkel day trip      │
│  │                         for two", "category": "trip_cost", "amount": 289.0}, {"description": "Airport        │
│  │                         parking, prepaid", "category": "trip_cost", "amount": 96.0}], "amount_claimed":      │
│  │                         1835.0, "documents": [{"name": "Skylark cancellation notice", "type": "letter",      │
│  │                         "content": "Skylark Airways, July 11, 2026. All Skylark flights between Miami and    │
│  │                         Nassau on July 12, 13 and 14 are canceled because of Hurricane Marisol. Booking      │
│  │                         SK-77Q41 for Gloria and Ramon Benitez is canceled and a travel voucher of $612.00    │
│  │                         has been issued."}, {"name": "Resort letter", "type": "letter", "content": "Coral    │
│  │                         Harbour Resort, Nassau, July 11, 2026. The booking for July 12 to 19 is canceled     │
│  │                         at the guest's request. The deposit of $1,450.00 is not refundable."}, {"name":      │
│  │                         "Receipts", "type": "receipt", "content": "Island Hoppers Tours, 2026-07-10:         │
│  │                         catamaran day trip for two $289.00, not refundable. Miami Airport Parking,           │
│  │                         2026-07-10: prepaid parking $96.00."}], "status": "open"}                            │
│  │     Actual Output:      {"decision": "refer", "payout": 0.0, "rationale": "Refer for review under section    │
│  │                         2: the carrier confirms Hurricane Marisol canceled service for three days, which     │
│  │                         meets the 24-hour trigger, and the resort and tour costs are documented as           │
│  │                         nonrefundable. The parking receipt does not establish that the $96 cost is           │
│  │                         nonrefundable, and the file does not establish whether the storm was publicly        │
│  │                         known before purchase, an exclusion under section 9; these unresolved points         │
│  │                         affect eligibility.", "cost": 0.0006856400000000001, "retrieved": ""}                │
│  │     Expected Output:    {"decision": "decline", "payo

⚠ WARNING: No hyperparameters logged.
» ]8;id=9280670;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.1s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 40.0% | Passed: 2 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [27]:
show_confusion_matrix(payout_table, "Payout tool")

In [35]:
len(payout_table[~payout_table["right"]])

3

In [36]:
len(payout_table)

5

### Experiment 2: the payout tool and memory

The same agent, now able to search its memory. Each run takes about 4 minutes.

In [29]:
memory_table = await run_experiment("Payout tool and memory", memory_agent)


CLM-1301: decline $0.00


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

CLM-1307: approve $562.95


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

CLM-1314: decline $0.00


...

✨ You're running DeepEval's latest Right Metric! (using None, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1301 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1307 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1314 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1317 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1320 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric        ┃ Average Score             ┃ Pass Rate                                            ┃ Total       │
│ ━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Right         │ 1.00                      │ 100.00% | passed=5 | failed=0                        │ 5           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=9280672;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.07s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 100.0% | Passed: 5 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [30]:
show_confusion_matrix(memory_table, "Payout tool and memory")

In [ ]:
memory_table[~memory_table["right"]]

If you use Confident AI, open the traces for this experiment to see what the agent searched for and what came back.


### Comparing the experiments

In [ ]:
pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=["Payout tool", "Payout tool and memory"]).round(3)

DeepEval also prints a table for each `evaluate` call, and keeps the results of the latest run in a `.deepeval` folder. On Confident AI, the test runs are under **Evals**, where you can compare them side by side, claim by claim.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Agentic RAG allows you to give an agent expertise about your business, and to build up its knowledge over time (long-term memory).
            Consider the expertise that you could equip an agent with for your business.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: replace Qdrant with a simple word lookup. Write a tool that searches the facts in <code>data/knowledge.json</code> for the words in its query, give it to the memory agent in place of <code>qdrant-find</code>, and run Experiment 2 again. <code>decide</code> picks out the search results by the tool's name, so change <code>qdrant-find</code> there to the name of your tool. How do the scores compare with searching by meaning?<br/><br/>
            Then give the agent a memory about each customer: a Markdown file for each one, named by the claim's <code>customer_id</code>, with a tool to read a customer's notes and a tool to add to them. Each customer has only one claim in this set, so write a few notes yourself to start with. With <code>N_TIMES</code> above 1, what should happen to the notes between runs?<br/><br/>
            Stretch challenge: move the retrieval into a sub-agent that has <code>qdrant-find</code> and the customer memory tools, and let the UnderCover agent call the sub-agent as a tool.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>